# Estudo de Caso 03: Comparativo Rigoroso: PPO vs. DQN Padrão

> **HaxBall RL Suite** - Comparação entre abordagem On-Policy Ator-Crítico e abordagem Off-Policy baseada em valor (*Value-Based*).

Neste notebook, executamos um benchmark experimental comparando:
1. **PPO (Proximal Policy Optimization):** Controle contínuo $\mathcal{A} \in [-1, 1]^3$ com Generalized Advantage Estimation (GAE-$\lambda$) e corte de gradiente $\epsilon = 0.20$.
2. **DQN (Deep Q-Network Padrão):** Espaço discreto de 18 ações com Replay Buffer de transições e Target Network sincronizada periodicamente.
3. Curvas de Recompensa Média, Taxa de Vitórias, Perda (*Loss*) e estabilidade em Auto-Confronto (*Self-Play*).


In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.insert(0, module_path)

from haxball.gym_env.haxball_env import HaxBallEnv
from haxball.bots.wall_rebound_bot import WallReboundBot
from haxball.rl.algorithms.ppo.ppo_trainer import PPOTrainer
from haxball.rl.algorithms.standard_rl.dqn_trainer import DQNTrainer

print('Ambientes e treinadores prontos para benchmark!')


## 1. Executando o Benchmark Experimental (5 Iterações Cada)

Ambos os algoritmos enfrentam o mesmo oponente padrão (`WallReboundBot` - especialista em tabelas) no estádio **Futsal 2v2**.


In [ ]:
map_file = os.path.join('..', 'haxball', 'maps', 'futsal_2v2.hbs')
bot = WallReboundBot('WallReboundBot')

# 1. Treinamento PPO (Contínuo)
print('=== INICIANDO BENCHMARK PPO ===')
env_ppo = HaxBallEnv(stadium_file=map_file, opponent_bot=bot, discrete_actions=False, max_steps=400)
ppo = PPOTrainer(env=env_ppo, policy_type='mlp', num_steps=512, batch_size=64, update_epochs=4)

ppo_rewards, ppo_winrates, ppo_losses = [], [], []
for it in range(1, 6):
    stats = ppo.train_iteration()
    ppo_rewards.append(stats['mean_reward'])
    ppo_winrates.append(stats['win_rate'])
    ppo_losses.append(stats['loss'])
    print(f'PPO Iter {it}: Reward = {stats["mean_reward"]:+.2f} | WinRate = {stats["win_rate"]:.1f}% | Loss = {stats["loss"]:.4f}')

# 2. Treinamento DQN (Discreto)
print('\n=== INICIANDO BENCHMARK DQN ===')
env_dqn = HaxBallEnv(stadium_file=map_file, opponent_bot=bot, discrete_actions=True, max_steps=400)
dqn = DQNTrainer(env=env_dqn, batch_size=32, target_update_freq=200)

dqn_rewards, dqn_losses = [], []
for it in range(1, 6):
    stats = dqn.train_step(num_env_steps=512)
    dqn_rewards.append(stats['mean_reward'])
    dqn_losses.append(stats['loss'])
    print(f'DQN Step {it*512}: Reward = {stats["mean_reward"]:+.2f} | Loss = {stats["loss"]:.4f} | Eps = {stats["epsilon"]:.2f}')


## 2. Análise Gráfica Comparativa
Comparação direta da convergência de recompensa e função de perda:


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

iterations = range(1, len(ppo_rewards) + 1)

# Gráfico 1: Recompensa Média
ax1.plot(iterations, ppo_rewards, 'o-', color='#2196F3', linewidth=2.5, label='PPO (Ator-Crítico Contínuo)')
ax1.plot(iterations, dqn_rewards, 's--', color='#FF5722', linewidth=2.5, label='DQN (Value-Based Discreto)')
ax1.set_title('Convergência de Recompensa Média', fontsize=12, fontweight='bold')
ax1.set_xlabel('Iteração de Treino')
ax1.set_ylabel('Recompensa Média por Episódio')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gráfico 2: Erro de Perda (Loss)
ax2.plot(iterations, ppo_losses, 'o-', color='#2196F3', linewidth=2.5, label='Perda PPO (Clipped Surrogate)')
ax2.plot(iterations, dqn_losses, 's--', color='#FF5722', linewidth=2.5, label='Perda DQN (TD Error Bellman)')
ax2.set_title('Comportamento da Função de Perda (Loss)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Iteração de Treino')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Conclusão do Estudo Comparativo

| Critério | PPO (Ator-Crítico) | DQN Padrão (Value-Based) |
|---|---|---|
| **Controle de Micro-Ângulos** | Fluido e analógico | Escalonado em passos de 45° |
| **Estabilidade em Self-Play** | Altíssima (updates conservadores) | Moderada/Baixa (buffer corrompido) |
| **Eficiência Amostral** | On-policy (descarta rollouts) | Off-policy (reutiliza buffer) |
| **Dribles Finos & Tabelas** | Aprendizado suave e progressivo | Frequentes oscilações por ação gulosa |
